In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

DRIVE_ROOT = "/content/drive/MyDrive"
PROJECT_ROOT = f"{DRIVE_ROOT}/OldPhotoRestoreProject"
REPO_DIR = f"{PROJECT_ROOT}/Bringing-Old-Photos-Back-to-Life"
INPUT_DIR = f"{PROJECT_ROOT}/input_images"
OUTPUT_DIR = f"{PROJECT_ROOT}/output_images"

os.makedirs(PROJECT_ROOT, exist_ok=True)
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("REPO_DIR:", REPO_DIR)
print("INPUT_DIR:", INPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
%cd /content/drive/MyDrive/OldPhotoRestoreProject

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/microsoft/Bringing-Old-Photos-Back-to-Life.git
else:
    print("Repo already exists in Drive.")


In [ ]:
%cd /content/drive/MyDrive/OldPhotoRestoreProject/Bringing-Old-Photos-Back-to-Life

In [ ]:
!pip install -r requirements.txt
!pip install lpips

In [ ]:
import os
import shutil

# For Face_Enhancement
fe_sync_batchnorm_target_path = "Face_Enhancement/models/networks/sync_batchnorm"
fe_cloned_repo_path = "Face_Enhancement/models/networks/Synchronized-BatchNorm-PyTorch"

if not os.path.exists(fe_sync_batchnorm_target_path):
    print("Face_Enhancement sync_batchnorm not found. Attempting to re-clone and copy.")
    if os.path.exists(fe_cloned_repo_path):
        print(f"Removing existing incomplete clone: {fe_cloned_repo_path}")
        shutil.rmtree(fe_cloned_repo_path)

    # Perform a fresh clone
    !git clone https://github.com/vacancy/Synchronized-BatchNorm-PyTorch {fe_cloned_repo_path}

    # Check if the required folder exists after cloning and then copy
    if os.path.exists(f"{fe_cloned_repo_path}/sync_batchnorm"):
        !cp -rf {fe_cloned_repo_path}/sync_batchnorm {os.path.dirname(fe_sync_batchnorm_target_path)}
        print("Successfully re-cloned and copied sync_batchnorm for Face_Enhancement.")
    else:
        print(f"Warning: 'sync_batchnorm' directory not found inside {fe_cloned_repo_path} after re-cloning. Clone might still be incomplete.")
else:
    print("Face_Enhancement sync_batchnorm already exists.")

# For Global (original code remains, as it seems to have worked based on previous output)
if not os.path.exists("Global/detection_models/sync_batchnorm"):
    !git clone https://github.com/vacancy/Synchronized-BatchNorm-PyTorch Global/detection_models/Synchronized-BatchNorm-PyTorch
    !cp -rf Global/detection_models/Synchronized-BatchNorm-PyTorch/sync_batchnorm Global/detection_models/
else:
    print("Global sync_batchnorm already exists.")

In [ ]:
import os

landmark_path = "Face_Detection/shape_predictor_68_face_landmarks.dat"

if not os.path.exists(landmark_path):
    %cd Face_Detection
    !wget http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
    !bzip2 -d shape_predictor_68_face_landmarks.dat.bz2
    %cd ..
else:
    print("Landmark model already exists.")

In [ ]:
import os

if not os.path.exists("Face_Enhancement/checkpoints"):
    %cd Face_Enhancement
    !wget https://github.com/microsoft/Bringing-Old-Photos-Back-to-Life/releases/download/v1.0/face_checkpoints.zip
    !unzip -o face_checkpoints.zip
    %cd ..
else:
    print("Face checkpoints already exist.")

if not os.path.exists("Global/checkpoints"):
    %cd Global
    !wget https://github.com/microsoft/Bringing-Old-Photos-Back-to-Life/releases/download/v1.0/global_checkpoints.zip
    !unzip -o global_checkpoints.zip
    %cd ..
else:
    print("Global checkpoints already exist.")

In [ ]:
paths = [
    "Face_Enhancement/checkpoints",
    "Global/checkpoints",
    "Face_Detection/shape_predictor_68_face_landmarks.dat",
    "Face_Enhancement/models/networks/sync_batchnorm",
    "Global/detection_models/sync_batchnorm",
]

for p in paths:
    print(p, "->", os.path.exists(p))

In [ ]:
# FIX: Install missing sync_batchnorm
import subprocess, shutil, os

REPO_DIR = "/content/drive/MyDrive/OldPhotoRestoreProject/Bringing-Old-Photos-Back-to-Life"

subprocess.run(["git", "clone", "https://github.com/vacancy/Synchronized-BatchNorm-PyTorch.git", "/tmp/syncbn"], capture_output=True)

sync_dst = f"{REPO_DIR}/Face_Enhancement/models/networks/sync_batchnorm"
if os.path.exists(sync_dst):
    shutil.rmtree(sync_dst)
shutil.copytree("/tmp/syncbn/sync_batchnorm", sync_dst)
print("✓ Done:", os.listdir(sync_dst))

In [ ]:
# FIX: Patch the import line in normalization.py
norm_file = f"{REPO_DIR}/Face_Enhancement/models/networks/normalization.py"

with open(norm_file, 'r') as f:
    content = f.read()

content = content.replace(
    "from models.networks.sync_batchnorm import SynchronizedBatchNorm2d",
    "from sync_batchnorm import SynchronizedBatchNorm2d"
)

with open(norm_file, 'w') as f:
    f.write(content)

print("✓ normalization.py patched")

In [ ]:
import os, glob, shutil, subprocess
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt
import torch

REPO_DIR = "/content/drive/MyDrive/OldPhotoRestoreProject/Bringing-Old-Photos-Back-to-Life"
INPUT_DIR = "/content/drive/MyDrive/OldPhotoRestoreProject/input_images"
OUTPUT_DIR = "/content/drive/MyDrive/OldPhotoRestoreProject/output_images"

# ── Step 1: Clear input folder
for f in glob.glob(f"{INPUT_DIR}/*"):
    os.remove(f)

# ── Step 2: Upload
print("Upload your damaged image...")
uploaded = files.upload()

uploaded_name = list(uploaded.keys())[0]
clean_name = "input_image.png"  # fixed name, no spaces/brackets issues
dst_path = f"{INPUT_DIR}/{clean_name}"

with open(dst_path, "wb") as f:
    f.write(uploaded[uploaded_name])

# ── Step 3: RESIZE IMMEDIATELY before anything else
img = Image.open(dst_path).convert("RGB")
w, h = img.size
print(f"Original size: {w} x {h}")

MAX_SIZE = 600  # conservative — safe for free Colab T4
if w > MAX_SIZE or h > MAX_SIZE:
    ratio = min(MAX_SIZE/w, MAX_SIZE/h)
    new_w, new_h = int(w * ratio), int(h * ratio)
    img = img.resize((new_w, new_h), Image.LANCZOS)
    img.save(dst_path)
    print(f"✓ Resized to: {new_w} x {new_h}")
else:
    print(f"✓ Size fine: {w} x {h}")

# ── Step 4: Clear GPU memory
torch.cuda.empty_cache()
print("✓ GPU cache cleared")

# ── Step 5: Verify file is actually small now
file_size_kb = os.path.getsize(dst_path) / 1024
print(f"✓ File size after resize: {file_size_kb:.0f} KB")

# ── Step 6: Run restoration
single_output_dir = f"{OUTPUT_DIR}/run_fixed"
if os.path.exists(single_output_dir):
    shutil.rmtree(single_output_dir)
os.makedirs(single_output_dir, exist_ok=True)

env = os.environ.copy()
env["PYTHONPATH"] = (
    f"{REPO_DIR}/Face_Enhancement:"
    f"{REPO_DIR}/Face_Enhancement/models/networks:"
    + env.get("PYTHONPATH", "")
)
env["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

cmd = [
    "python", "run.py",
    "--input_folder", INPUT_DIR,
    "--output_folder", single_output_dir,
    "--GPU", "0",
    "--with_scratch"
]

print("\nRunning restoration pipeline...")
result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True, env=env)

print("RETURN CODE:", result.returncode)
print("STDOUT:", result.stdout[-3000:])
print("STDERR:", result.stderr[-1500:])

# ── Step 7: Show result
final_dir = f"{single_output_dir}/final_output"
imgs = []
for ext in ["*.png","*.jpg","*.jpeg","*.PNG","*.JPG"]:
    imgs.extend(glob.glob(f"{final_dir}/{ext}"))

if imgs:
    orig = Image.open(dst_path).convert("RGB")
    rest = Image.open(imgs[0]).convert("RGB")

    plt.figure(figsize=(14, 6))
    plt.subplot(1,2,1); plt.imshow(orig); plt.title("Damaged Input"); plt.axis("off")
    plt.subplot(1,2,2); plt.imshow(rest); plt.title("Restored Output"); plt.axis("off")
    plt.tight_layout()
    plt.show()
    print("✓ Restored saved at:", imgs[0])
else:
    print("\nNo final output found. All files produced:")
    for root, _, fs in os.walk(single_output_dir):
        for fn in fs:
            print(" ", os.path.join(root, fn))

In [ ]:
!pip install pyngrok fastapi uvicorn python-multipart nest-asyncio

In [ ]:
import os
os.chdir('/content/drive/MyDrive/OldPhotoRestoreProject/Bringing-Old-Photos-Back-to-Life')

# Upload a test image to /tmp/test_input/
os.makedirs('/tmp/test_input', exist_ok=True)
os.makedirs('/tmp/test_output', exist_ok=True)

# Copy any one image to test
# (put a real old photo path here)
os.system("cp /content/your_test_photo.jpg /tmp/test_input/")

# Run the full pipeline
os.system("""
python run.py \
  --input_folder /tmp/test_input \
  --output_folder /tmp/test_output \
  --GPU 0 \
  --with_scratch
""")

# Check output
for f in os.listdir('/tmp/test_output'):
    print(f)

In [ ]:
# ══════════════════════════════════════════════════════════════
#  Old Photo Restoration  |  FastAPI + ngrok Server
# ══════════════════════════════════════════════════════════════

import os, re, uuid, glob, subprocess, io
import nest_asyncio
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, JSONResponse
from pyngrok import ngrok
from PIL import Image
import uvicorn

nest_asyncio.apply()

# ── Config ────────────────────────────────────────────────────
REPO     = "/content/drive/MyDrive/OldPhotoRestoreProject/Bringing-Old-Photos-Back-to-Life"
MAX_SIZE = 512
# Paste your token from https://dashboard.ngrok.com/get-started/your-authtoken
# Do not commit a real token to GitHub.
NGROK_TOKEN = "YOUR_NGROK_AUTHTOKEN"

# ── Patch numpy casting bug in repo ───────────────────────────
DLIB_FILE = f"{REPO}/Face_Detection/align_warp_back_multiple_dlib.py"

with open(DLIB_FILE, "r") as f:
    src = f.read()

# Fix: mask *= 255.0  →  mask = (mask * 255.0).astype('uint8')
patched = src.replace(
    "mask *= 255.0",
    "mask = (mask * 255.0).astype('uint8')"
)

if patched != src:
    with open(DLIB_FILE, "w") as f:
        f.write(patched)
    print("✓ Patched numpy casting bug in align_warp_back_multiple_dlib.py")
else:
    print("✓ Patch already applied or line not found — skipping")

# ── FastAPI App ───────────────────────────────────────────────
app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_origin_regex=".*",
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
    expose_headers=["*"],
)

@app.get("/")
def root():
    return {"status": "✅ Old Photo Restoration API is running!"}

@app.post("/restore")
async def restore(file: UploadFile = File(...)):
    job_id     = str(uuid.uuid4())
    input_dir  = f"/tmp/{job_id}/input"
    output_dir = f"/tmp/{job_id}/output"
    os.makedirs(input_dir, exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)

    # ── Clean filename ────────────────────────────────────────
    clean_name = re.sub(r'[^a-zA-Z0-9_.]', '_', file.filename).replace("..", ".")
    input_path = f"{input_dir}/{clean_name}"

    # ── Resize & save ─────────────────────────────────────────
    content = await file.read()
    img = Image.open(io.BytesIO(content)).convert("RGB")
    w, h = img.size
    print(f"  Original size : {w} × {h}")

    if w > MAX_SIZE or h > MAX_SIZE:
        ratio = min(MAX_SIZE / w, MAX_SIZE / h)
        img = img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)

    img.save(input_path)
    print(f"  Saved as      : {input_path}  ({img.size[0]} × {img.size[1]})")

    # ── Environment ───────────────────────────────────────────
    env = os.environ.copy()
    env["PYTHONPATH"] = (
        f"{REPO}/Face_Enhancement:"
        f"{REPO}/Face_Enhancement/models/networks:"
        + env.get("PYTHONPATH", "")
    )
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    # ── Run pipeline ──────────────────────────────────────────
    cmd = [
        "python", "run.py",
        "--input_folder",  input_dir,
        "--output_folder", output_dir,
        "--GPU",           "0",
        "--with_scratch"
    ]

    print(f"\n  Running: {' '.join(cmd)}\n")

    result = subprocess.run(
        cmd, cwd=REPO, capture_output=True,
        text=True, env=env, timeout=300
    )

    print(f"  Return code : {result.returncode}")
    if result.stdout.strip():
        print(f"  STDOUT      :\n{result.stdout[-2000:]}")
    if result.stderr.strip():
        print(f"  STDERR      :\n{result.stderr[-1000:]}")

    if result.returncode != 0:
        return JSONResponse(status_code=500, content={
            "error":  result.stderr[-2000:],
            "stdout": result.stdout[-1000:]
        })

    # ── Find output ───────────────────────────────────────────
    final_dir = f"{output_dir}/final_output"
    imgs = []
    for ext in ["*.png", "*.jpg", "*.jpeg", "*.PNG", "*.JPG"]:
        imgs.extend(glob.glob(f"{final_dir}/{ext}"))

    if not imgs:
        return JSONResponse(status_code=500, content={
            "error": "No output generated",
            "log":   result.stdout[-1000:]
        })

    print(f"\n  ✅ Restored: {imgs[0]}")

    return FileResponse(
        imgs[0],
        media_type="image/png",
        filename="restored.png",
        headers={
            "Access-Control-Allow-Origin": "*",
            "ngrok-skip-browser-warning":  "true"
        }
    )

# ── Start ngrok + server ──────────────────────────────────────
ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()

public_url = ngrok.connect(8000)

print("\n" + "═" * 50)
print(f"  🚀 API URL: {public_url.public_url}")
print("═" * 50 + "\n")

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()